# Songo — AlphaZero v3 (Colab)

Pipeline : ResNet 2D 6.7M params + EGTB distillation + self-play MCTS batché.

**Trois phases** :
1. **Pretrain** depuis EGTB (one-shot, ~30 min). Skip si déjà fait.
2. **Self-play training** (résumable, plusieurs sessions Colab).
3. **Export ONNX** + téléchargement.

**Résumé d'une session interrompue** : remet `RESUME = True`, relance les cellules 1–4 puis la cellule training. `train_v3.py` skipe automatiquement les itérations dont `history.json` existe déjà.

## Prérequis (à faire sur ton PC une seule fois)

```powershell
# 1. Générer un dataset EGTB de distillation
cd D:\akong-online\engine-rs
cargo run --release --bin egtb_dump_samples -- 4 ../egtb-data/samples-n4.jsonl
python ../training/distillation.py ../egtb-data/samples-n4.jsonl ../egtb-data/samples-n4.npz

# 2. Zipper le code Python
cd D:\akong-online
Compress-Archive -Path training\*.py, training\requirements.txt -DestinationPath training_v3.zip -Force

# 3. Uploader sur Google Drive (manuellement)
#    → MyDrive/songo_alphazero/training_v3.zip
#    → MyDrive/songo_alphazero/egtb-data/samples-n4.npz
```

In [ ]:
# Cell 1 — GPU + CPU
import torch, os

print(f'PyTorch        : {torch.__version__}')
print(f'CPU cores      : {os.cpu_count()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️  No GPU. Runtime > Change runtime type > GPU')


In [ ]:
# Cell 2 — Mount Drive + paths
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR  = '/content/drive/MyDrive/songo_alphazero'
RUN_DIR    = f'{DRIVE_DIR}/runs/v3'                  # train_v3.py state lives here
PRETRAIN_DIR = f'{DRIVE_DIR}/checkpoints/pretrain-v3'
EGTB_NPZ   = f'{DRIVE_DIR}/egtb-data/samples-n4.npz'  # pre-uploaded
ONNX_OUT   = f'{DRIVE_DIR}/songo_nn_v3.onnx'
CODE_ZIP   = f'{DRIVE_DIR}/training_v3.zip'           # pre-uploaded
WORK_DIR   = '/content/training'

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(PRETRAIN_DIR, exist_ok=True)

print(f'DRIVE_DIR    = {DRIVE_DIR}')
print(f'RUN_DIR      = {RUN_DIR}')
print(f'PRETRAIN_DIR = {PRETRAIN_DIR}')
print(f'EGTB_NPZ     = {EGTB_NPZ} ({"OK" if os.path.exists(EGTB_NPZ) else "MISSING"})')
print(f'CODE_ZIP     = {CODE_ZIP} ({"OK" if os.path.exists(CODE_ZIP) else "MISSING"})')

# Show existing checkpoints (for sanity)
if os.path.exists(RUN_DIR):
    iters = sorted([d for d in os.listdir(RUN_DIR) if d.startswith('iter-')])
    print(f'\n[run] {len(iters)} iteration dirs in {RUN_DIR}')
    if iters:
        print(f'      latest: {iters[-1]}')


In [ ]:
# Cell 3 — Sync code from Drive (or upload if missing)
import os, shutil, zipfile

if not os.path.exists(CODE_ZIP):
    # First-time: upload manually
    from google.colab import files
    print('⚠️  CODE_ZIP not found on Drive — upload training_v3.zip now')
    uploaded = files.upload()
    name = list(uploaded.keys())[0]
    shutil.copy2(name, CODE_ZIP)
    print(f'✓ cached to {CODE_ZIP}')

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

with zipfile.ZipFile(CODE_ZIP, 'r') as z:
    names = z.namelist()
    has_subfolder = any(n.startswith('training/') for n in names)
    z.extractall('/content/' if has_subfolder else WORK_DIR)

essential = ['network_v3.py', 'encoding_v3.py', 'distillation.py',
             'pretrain_from_egtb.py', 'self_play_v3.py', 'arena_v3.py',
             'train_v3.py', 'export_onnx_v3.py']
missing = [f for f in essential if not os.path.exists(f'{WORK_DIR}/{f}')]
if missing:
    raise RuntimeError(f'missing files in zip: {missing}')
print(f'✓ code synced to {WORK_DIR}')


In [ ]:
# Cell 4 — Install deps (torch already on Colab)
!pip install -q onnx onnxruntime tqdm 2>&1 | tail -3
print('✓ deps OK')


---
## Phase 1 — Pretrain depuis EGTB

One-shot. Skip si `model_best.pt` existe déjà dans `PRETRAIN_DIR`.

In [ ]:
# Cell 5 — Pretrain (skipped if checkpoint exists)
import os, sys, subprocess

PRETRAIN_CKPT = f'{PRETRAIN_DIR}/model_best.pt'

if os.path.exists(PRETRAIN_CKPT):
    print(f'✓ pretrain checkpoint already exists — SKIPPING')
    print(f'   {PRETRAIN_CKPT}')
elif not os.path.exists(EGTB_NPZ):
    print(f'⚠️  EGTB samples not found on Drive: {EGTB_NPZ}')
    print('   Upload egtb-data/samples-n4.npz to Drive, then re-run this cell.')
else:
    cmd = [
        'python', f'{WORK_DIR}/pretrain_from_egtb.py',
        '--data',   EGTB_NPZ,
        '--out',    PRETRAIN_DIR,
        '--epochs', '20',
        '--batch',  '512',
        '--lr',     '1e-3',
        '--device', 'cuda',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=WORK_DIR)


---
## Phase 2 — Self-play training (résumable)

Chaque itération : self-play (champion vs lui-même) → mix EGTB → train candidate → arena → promotion si Wilson CI > 0.55.

Les checkpoints + npz vont sur Drive (`RUN_DIR`). En cas de coupure, les itérations terminées sont skippées au redémarrage.

In [ ]:
# Cell 6 — Training config
NUM_ITERATIONS  = 30      # how many to attempt this session (resumes from last)
SELFPLAY_GAMES  = 80      # per iteration
SELFPLAY_SIMS   = 200     # MCTS sims per move during self-play
BUFFER_ITERS    = 5       # keep last N iterations of self-play in the buffer

EPOCHS_PER_ITER = 4
BATCH_SIZE      = 512
LR              = 2e-4
EGTB_RATIO      = 0.3     # 30% of training batches drawn from EGTB

ARENA_GAMES     = 40
ARENA_SIMS      = 120
WIN_THRESHOLD   = 0.55

print(f'iterations      : {NUM_ITERATIONS}')
print(f'selfplay/iter   : {SELFPLAY_GAMES} games × {SELFPLAY_SIMS} sims')
print(f'arena/iter      : {ARENA_GAMES} games × {ARENA_SIMS} sims')
print(f'EGTB ratio      : {EGTB_RATIO}')

# Estimate (very rough — GPU-dependent)
est_min_per_iter = SELFPLAY_GAMES * 0.5 + EPOCHS_PER_ITER * 0.5 + ARENA_GAMES * 0.3
print(f'\n~{est_min_per_iter:.0f} min/iter on A100 (rough estimate)')
print(f'~{est_min_per_iter * NUM_ITERATIONS / 60:.1f}h for {NUM_ITERATIONS} iterations')


In [ ]:
# Cell 7 — Run training (resumable)
import subprocess, os

args = [
    'python', f'{WORK_DIR}/train_v3.py',
    '--run-dir',         RUN_DIR,
    '--iterations',      str(NUM_ITERATIONS),
    '--selfplay-games',  str(SELFPLAY_GAMES),
    '--selfplay-sims',   str(SELFPLAY_SIMS),
    '--buffer-iters',    str(BUFFER_ITERS),
    '--epochs',          str(EPOCHS_PER_ITER),
    '--batch',           str(BATCH_SIZE),
    '--lr',              str(LR),
    '--egtb-ratio',      str(EGTB_RATIO),
    '--arena-games',     str(ARENA_GAMES),
    '--arena-sims',      str(ARENA_SIMS),
    '--win-threshold',   str(WIN_THRESHOLD),
    '--device',          'cuda',
]

if os.path.exists(EGTB_NPZ):
    args += ['--egtb', EGTB_NPZ]
else:
    print('⚠️  EGTB samples missing — training without distillation (slower convergence)')

champion = f'{RUN_DIR}/champion.pt'
pretrain = f'{PRETRAIN_DIR}/model_best.pt'
if not os.path.exists(champion) and os.path.exists(pretrain):
    args += ['--init', pretrain]
    print(f'[bootstrap] using pretrain checkpoint as initial champion')

print('Running:', ' '.join(args))
print('-' * 60)
subprocess.run(args, check=True, cwd=WORK_DIR)


---
## Phase 3 — Export ONNX + télécharger

In [ ]:
# Cell 8 — Export champion → ONNX
import subprocess, os

champion = f'{RUN_DIR}/champion.pt'
if not os.path.exists(champion):
    raise RuntimeError(f'champion checkpoint not found: {champion}')

cmd = [
    'python', f'{WORK_DIR}/export_onnx_v3.py',
    '--ckpt', champion,
    '--out',  ONNX_OUT,
]
subprocess.run(cmd, check=True, cwd=WORK_DIR)

size_mb = os.path.getsize(ONNX_OUT) / 1e6
print(f'\n✓ {ONNX_OUT} ({size_mb:.1f} MB)')


In [ ]:
# Cell 9 — Download ONNX
from google.colab import files
import os
if os.path.exists(ONNX_OUT):
    files.download(ONNX_OUT)
    print('→ place in public/songo_nn_v3.onnx of the web project')
else:
    print(f'❌ not found: {ONNX_OUT} — run cell 8 first')


---
## Résumer une session interrompue

1. Runtime > Change runtime type > GPU (A100 si Pro+)
2. Run cell 1–4 (GPU check, Drive, code sync, deps)
3. **Skip cell 5** (pretrain déjà fait) ou laisse-le — il skipe automatiquement
4. Run cell 6–7 — `train_v3.py` skipe les itérations déjà dans `RUN_DIR/iter-NNN/history.json`

## Quoi regarder pendant le training

- **arena.json** : `wilson_lower` doit grimper vers 0.55+ pour promote
- **history.json** : `total` (loss totale) doit baisser monotone sur les premières itérations
- **global_history.json** : résumé toutes itérations

Si tu vois 5 échecs d'arena consécutifs :
- Soit le champion a convergé (= bon, plus rien à apprendre dans ces conditions)
- Soit le LR est trop élevé ou la température trop basse — baisse `LR` à 1e-4 ou augmente `SELFPLAY_GAMES`
